### Build Results Fact

In [0]:
%run ../00-common/01-environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

### 1. Read Source Table

In [0]:
from pyspark.sql import functions as F

results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
    .withColumnsRenamed(
        {
            "finish_position": "final_position",
            "finish_position_text": "final_position_text",
        }
    )
    .withColumn("session_type", F.lit("RACE"))
    .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
    .withColumnsRenamed(
        {
            "finish_position": "final_position",
            "finish_position_text": "final_position_text",
        }
    )
    .withColumn("session_type", F.lit("SPRINT"))
    .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)
display(results_df)
display(sprints_df)

### 2. Joining Two Tables

In [0]:
results_sprints_df = (
    results_df
    .unionByName(sprints_df)
)
display(results_sprints_df)

### 3. Adding Derived columns 

In [0]:
results_sprints_df = (
     results_sprints_df.withColumn("is_win", F.col("final_position") == 1)
    .withColumn("is_podium", F.col("final_position").between(1, 3))
    .withColumn("has_points", F.col("points") > 0)
)

### Writing data to gold table

In [0]:
(
    results_sprints_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(target_table)
    )
display(spark.table(target_table))